# 14 - D17: more field photos for training

**In short:** field photos from new places are checked and added to the **training data
only**, as a new list `train_d17.csv`. Nothing the current model (v1) depends on changes:
`train.csv`, validation, test, the selection set, PlantDoc and every model folder stay as
they are (v1 was also backed up on 20 Oct to `backups/v1_2026-10-20`). Notebook 15 then
retrains ResNet-50 on `train_d17.csv`, and a rule fixed before training decides whether
the new model is adopted.

| Source | Crop | Classes used | What is used | Licence |
|---|---|---|---|---|
| Pakistan (3mbnb82mxd v2) | tomato | early blight, healthy, late blight | only the authors' own phone photos (116); the rest are web images or augmented copies | CC BY 4.0 |
| Taiwan (ngdgg79rzb v1) | tomato | bacterial spot, healthy, late blight | `taiwan.7z` only; rotated, mirrored and brightened copies reduced to one per photo | CC BY 4.0 |
| Uttar Pradesh, India (3j4nfkvp2n v1) | potato | early blight, late blight, healthy | at most 1,500 per class, random with the project seed | CC BY 4.0 |
| Bangladesh 10-class (sb4wwjs8bz v1) | maize | northern leaf blight, common rust, gray leaf spot, healthy | archive put on Drive by hand (the API serves no files) | CC BY 4.0 |

**Checks:** the licence is read from Mendeley for every source; API downloads are verified
against Mendeley's SHA-256 checksums (the hand-downloaded maize archive's checksum is
recorded). Only unambiguous folders are mapped. A photo is dropped if it matches any
existing photo (training sources, PlantDoc or the selection set), exactly or as a
near-duplicate, also when turned or mirrored; near-copies inside the new photos are
reduced to one. **You then look at a sheet of sample photos per class before anything is
saved.**

**Before running:** the maize archive must be in
`MyDrive/CropDiseaseDiagnosis/datasets/raw/d17/maize_bd10/` (do not unzip it). If it is not
there yet, the notebook runs without maize and says so; run it again once it is there.

No GPU needed. About 45-75 minutes: the India potato archives are 8.7 GB (downloaded to
Colab's own disk, not Drive), then the duplicate checks (progress bars).

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

Mounted at /content/drive


In [ ]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

Already up to date.
Refresh index: 100% (294/294), done.
## main...origin/main
 M ml/notebooks/14_d17_field_training_data.ipynb
?? ml/results/fusion_quick_test/


In [ ]:
%pip install -q -r {ML_DIR}/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 2.7 MB/s eta 0:00:00


## Licence and download

Archives fetched by the API go to Colab's disk only. Kept photos are saved later, shrunk,
to `raw/d17_photos/` on Drive; the hand-downloaded maize archive stays in `raw/d17/`.

In [ ]:
import importlib

import pandas as pd

from src.data import cleaning, d17, selection_set
importlib.reload(cleaning); importlib.reload(selection_set); importlib.reload(d17)

MANUAL_DIR = config.RAW_DATA_DIR / "d17"          # hand-downloaded archives, on Drive
PHOTOS_DIR = config.RAW_DATA_DIR / "d17_photos"   # kept photos, on Drive (written at the end)
LOCAL = Path("/content/d17")                      # downloads and unpacked files, Colab disk only

# v1 must never be touched: this notebook writes only these places.
WRITES = [PHOTOS_DIR, config.SPLITS_DIR / "train_d17.csv", config.METADATA_DIR / "d17_sources.csv",
          config.METADATA_DIR / "d17_manifest.csv"]
assert config.SPLITS_DIR / "train.csv" not in WRITES and not any(config.MODELS_DIR in p.parents for p in WRITES)

records, skipped = [], []
for name, source in d17.SOURCES.items():
    licence = d17.check_licence(source)
    dest = MANUAL_DIR / name if source.manual else LOCAL / "archives" / name
    try:
        found = d17.download(name, dest)
    except FileNotFoundError as err:
        if not source.manual:
            raise
        print("SKIPPED:", err)
        skipped.append(name)
        continue
    for record in found:
        records.append({**record, "licence": licence, "dataset": f"{source.dataset_id} v{source.version}",
                        "citation": source.citation})
sources_table = pd.DataFrame(records)
display(sources_table[["source", "dataset", "licence", "file", "bytes", "sha256", "how"]])
print("Sources used:", sorted(sources_table["source"].unique()), "| skipped:", skipped or "none")

,source,dataset,licence,file,bytes,sha256,how
0,tomato_pakistan,3mbnb82mxd v2,CC BY 4.0,Tomato_Leaf_Disease_Dataset_Pakistan.zip,726457628,2a2b36a8f972337f6d0e95e436835e55ffee5291fa0727...,"Mendeley API, checksum verified"
1,tomato_taiwan,ngdgg79rzb v1,CC BY 4.0,taiwan.7z,48008752,f623d13a7f0388dada08ca5b5f22716557defce3832f50...,"Mendeley API, checksum verified"
2,potato_india,3j4nfkvp2n v1,CC BY 4.0,EB.zip,2766326616,cffd37bbb79e75c0e23c1486f88f0a7c873b3fe67f643c...,"Mendeley API, checksum verified"
3,potato_india,3j4nfkvp2n v1,CC BY 4.0,Healthy.zip,3683027938,2b7e4107d7ba03c0ef9636831aa4de7d333435df238f66...,"Mendeley API, checksum verified"
4,potato_india,3j4nfkvp2n v1,CC BY 4.0,LB.zip,2323338293,f4d31182b5d2f147c256e1c73838eac9b17592b89ed2b1...,"Mendeley API, checksum verified"
5,maize_bd10,sb4wwjs8bz v1,CC BY 4.0,An Open-Source Multi-Format Maize Leaf Image D...,3110426444,5bde8a33eb390434507e00366eb3847285798714bc0767...,hand-downloaded (API offers no files); checksu...


Sources used: ['maize_bd10', 'potato_india', 'tomato_pakistan', 'tomato_taiwan'] | skipped: none


## Unpack and map the folders

**Check this table before going on:** every folder, how many photos, and what it became.
For maize and Taiwan the folder names were not visible before download: if a folder that
clearly holds one of our classes says "unmapped", stop and tell Claude the folder name.

In [16]:
for record in records:
    folder = (MANUAL_DIR if record["how"].startswith("hand") else LOCAL / "archives") / record["source"]
    d17.extract(folder / record["file"], LOCAL / "unpacked" / record["source"])
used = [name for name in d17.SOURCES if name not in skipped]
inventory = pd.concat([d17.inventory(name, LOCAL / "unpacked" / name) for name in used], ignore_index=True)
summary = d17.folder_summary(inventory)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 200)
display(summary)

,source,folder,label_reason,images,mapped_to
0,maize_bd10,An Open-Source Multi-Format Maize Leaf Image Datas/maize (corn) leaf images/Annotation...,excluded: not one of our classes,100,None
1,maize_bd10,An Open-Source Multi-Format Maize Leaf Image Datas/maize (corn) leaf images/Annotation...,excluded: same photos as RAW_Image (annotation copy),100,None
2,maize_bd10,An Open-Source Multi-Format Maize Leaf Image Datas/maize (corn) leaf images/Annotation...,excluded: same photos as RAW_Image (annotation copy),200,None
3,maize_bd10,An Open-Source Multi-Format Maize Leaf Image Datas/maize (corn) leaf images/Annotation...,excluded: not one of our classes,100,None
4,maize_bd10,An Open-Source Multi-Format Maize Leaf Image Datas/maize (corn) leaf images/Annotation...,excluded: same photos as RAW_Image (annotation copy),100,None
5,maize_bd10,An Open-Source Multi-Format Maize Leaf Image Datas/maize (corn) leaf images/Annotation...,excluded: same photos as RAW_Image (annotation copy),94,None
6,maize_bd10,An Open-Source Multi-Format Maize Leaf Image Datas/maize (corn) leaf images/Annotation...,excluded: not one of our classes,99,None
7,maize_bd10,An Open-Source Multi-Format Maize Leaf Image Datas/maize (corn) leaf images/Augmented_...,"excluded: augmented copies, not independent photos",300,None
8,maize_bd10,An Open-Source Multi-Format Maize Leaf Image Datas/maize (corn) leaf images/Augmented_...,"excluded: augmented copies, not independent photos",405,None
9,maize_bd10,An Open-Source Multi-Format Maize Leaf Image Datas/maize (corn) leaf images/Augmented_...,"excluded: augmented copies, not independent photos",660,None


In [17]:
# Workaround for a bug found: the "already unpacked" marker files end in .zip, so the
# search for zips inside a zip picked up an earlier marker when a source has several archives.
# The same fix goes into the code in the next pull request.
import zipfile
def _extract_fixed(archive, out):
    marker = out / f".extracted_{archive.name}"
    if marker.exists():
        return
    out.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        z.extractall(out)
    inner = [p for p in out.rglob("*.zip") if not p.name.startswith(".extracted_")]
    for p in inner:
        _extract_fixed(p, p.with_suffix(""))
    marker.touch()
selection_set.extract = _extract_fixed
print("Patched. Now run the unpack cell above again.")


Patched. Now run the unpack cell above again.


## Duplicate checks and the potato cap

Existing photos: every training-source and PlantDoc photo (`clean_manifest.csv`, notebook
06) and every selection-set photo (`selection_manifest.csv`, notebook 12). The India potato
cap (1,500 per class) is applied after the checks, so it counts independent photos.

In [18]:
existing = pd.concat([
    pd.read_csv(config.METADATA_DIR / "clean_manifest.csv", usecols=["md5", "phash"]),
    pd.read_csv(config.METADATA_DIR / "selection_manifest.csv", usecols=["md5", "phash"]),
], ignore_index=True)
mapped = inventory[inventory["class"].notna()].reset_index(drop=True)
prints = selection_set.fingerprint_all(list(mapped["path"]))
checked = pd.concat([mapped, prints], axis=1)
unmapped = inventory[inventory["class"].isna()].assign(error=None, md5=None, phash=None, phash_variants=None,
                                                       width=None, height=None)
result = selection_set.deduplicate(pd.concat([checked, unmapped], ignore_index=True), existing,
                                   config.MIN_IMAGE_SIZE, config.NEAR_DUPLICATE_MAX_DISTANCE)
# A copy of a matched photo that was changed a little more must not slip through on its own.
result = d17.drop_copies_of_matches(result, config.NEAR_DUPLICATE_MAX_DISTANCE)
result["reason"] = result["reason"].replace("near_duplicate_within_selection", "near_duplicate_within_new_photos")
result = d17.apply_cap(result, d17.CAP_PER_CLASS, config.RANDOM_SEED)

reasons = (result.assign(reason=result["reason"].replace("", "kept"))
           .groupby(["source", "reason"]).size().rename("images").reset_index())
display(reasons)
kept = result[result["keep"]]
counts = kept.pivot_table(index="class", columns="source", values="path", aggfunc="count", fill_value=0)
counts["total"] = counts.sum(axis=1)
display(counts)

Fingerprinting:   0%|          | 0/16298 [00:00<?, ?it/s]

,source,reason,images
0,maize_bd10,excluded: ambiguous: northern or southern leaf blight,220
1,maize_bd10,"excluded: augmented copies, not independent photos",2982
2,maize_bd10,excluded: not one of our classes,1149
3,maize_bd10,excluded: same photos as RAW_Image (annotation copy),494
4,maize_bd10,excluded: same photos as RAW_Image (converted copy),569
5,maize_bd10,kept,108
6,maize_bd10,near_duplicate_within_new_photos,241
7,potato_india,copy_of_a_photo_matching_existing,2
8,potato_india,kept,3442
9,potato_india,near_duplicate_of_existing_photo,10


source,maize_bd10,potato_india,tomato_pakistan,tomato_taiwan,total
class,,,,,
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot,12,0,0,0,12
Corn_(maize)___Common_rust_,17,0,0,0,17
Corn_(maize)___healthy,79,0,0,0,79
Potato___Early_blight,0,1301,0,0,1301
Potato___Late_blight,0,1500,0,0,1500
Potato___healthy,0,641,0,0,641
Tomato___Bacterial_spot,0,0,0,110,110
Tomato___Early_blight,0,0,39,0,39
Tomato___Late_blight,0,0,7,98,105


## STOP: look at the photos

One sheet per source and class, 18 random kept photos each. Check that each sheet shows
real field (or garden) photos of that disease: not drawings, not text, not lab photos of a
single leaf on plain paper unless the whole source is like that, and not a different crop.
If a sheet looks wrong, stop and describe it to Claude. Nothing has been saved yet.

In [19]:
REVIEW_DIR = LOCAL / "review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
sheets = []
for (name, cls), paths in d17.review_sample(kept, 18, config.RANDOM_SEED).items():
    total = int(((kept["source"] == name) & (kept["class"] == cls)).sum())
    sheet = d17.contact_sheet(paths, f"{name} | {cls} | {total} kept")
    target = REVIEW_DIR / f"{name}__{cls.replace(' ', '_').replace('(', '').replace(')', '')}.jpg"
    sheet.save(target, quality=85)
    sheets.append(target)
    display(sheet)

Output hidden; open in https://colab.research.google.com to view.

In [21]:
# After looking at EVERY sheet above, change False to True and run this cell.
I_CHECKED_THE_PHOTOS = True

assert I_CHECKED_THE_PHOTOS, "Look at every sheet above first, then set I_CHECKED_THE_PHOTOS = True."
print("Checked. Saving next.")

Checked. Saving next.


## Save the kept photos and `train_d17.csv`

Photos are saved to Drive, shrunk to a shorter side of 256 pixels (the size every split is
shrunk to before the model sees it), as `raw/d17_photos/<source>/<class>/<file>`.
`train_d17.csv` is `train.csv` unchanged plus one row per new photo, with the same columns.

In [ ]:
import shutil

# A rerun keeps a different set of photos, so this notebook's own earlier copies go first.
for name in used:
    if (PHOTOS_DIR / name).exists():
        shutil.rmtree(PHOTOS_DIR / name)

rows, jobs = [], []
for i, row in kept.reset_index(drop=True).iterrows():
    safe = f"{i:05d}_{Path(row['path']).name}".replace(" ", "_")
    target = PHOTOS_DIR / row["source"] / row["class"] / safe
    jobs.append((Path(row["path"]), target))
    rows.append({"path": str(target.relative_to(config.RAW_DATA_DIR)), "class": row["class"], "source": row["source"],
                 "crop": row["crop"], "dup_group": int(row["dup_group"]), "md5": row["md5"]})
selection_set.save_all_small(jobs)
added = pd.DataFrame(rows)

train = pd.read_csv(config.SPLITS_DIR / "train.csv")
train_d17 = d17.build_train_d17(train, added)
train_d17.to_csv(config.SPLITS_DIR / "train_d17.csv", index=False)
sources_table.to_csv(config.METADATA_DIR / "d17_sources.csv", index=False)
result.drop(columns=["path"]).assign(path=result["path"].astype(str)).to_csv(
    config.METADATA_DIR / "d17_manifest.csv", index=False)

class_table = d17.class_counts(train, train_d17)
display(class_table)
print(f"{len(added)} photos added; train.csv {len(train)} rows (unchanged) -> train_d17.csv {len(train_d17)} rows")

Saving photos:   0%|          | 0/3976 [00:00<?, ?it/s]

## Copy the summaries into the repository

In [ ]:
REPO_RESULTS = ML_DIR / "results" / "d17_field_data"
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
sources_table.to_csv(REPO_RESULTS / "sources.csv", index=False)
summary.to_csv(REPO_RESULTS / "folders.csv", index=False)
reasons.to_csv(REPO_RESULTS / "checks.csv", index=False)
counts.to_csv(REPO_RESULTS / "counts.csv")
class_table.to_csv(REPO_RESULTS / "train_class_counts.csv")
(REPO_RESULTS / "review_sheets").mkdir(exist_ok=True)
for sheet in sheets:
    shutil.copy(sheet, REPO_RESULTS / "review_sheets" / sheet.name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))

In [ ]:
drive.flush_and_unmount()
drive.mount("/content/drive")
print("Drive synced and remounted.")

## Save to GitHub

Save the notebook first (Ctrl+S); only the results folder is committed, to a new branch.

In [ ]:
from google.colab import userdata

from src import github_sync

shutil.copy(ML_DIR / "notebooks" / "14_d17_field_training_data.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/d17-field-data",
    message="D17: check extra field photos and write train_d17.csv",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))

In [ ]:
print(len(records), len(inventory), int(kept.shape[0]))


In [ ]:
pd.set_option("display.max_colwidth", 300)
for folder, n, why in summary[summary["source"] == "maize_bd10"][["folder", "images", "label_reason"]].values:
    print(n, "|", folder.split("leaf images/")[-1], "|", why.split(":")[0])


In [ ]:
# Mapping fixes after reading the folder table (20 Oct); the same changes go into the code.
taiwan = d17.SOURCES["tomato_taiwan"]
taiwan.classes["health"] = "Tomato___healthy"            # Taiwan's healthy folder is called "health"
taiwan.excluded["augmentation"] = "augmented copies, not independent photos"   # keep "Preprocessed data" only

maize = d17.SOURCES["maize_bd10"]
maize.excluded["annotation"] = "same photos as RAW_Image (annotation copy)"
maize.excluded["converted"] = "same photos as RAW_Image (converted copy)"
maize.excluded["corn leaf blight"] = "ambiguous: northern or southern leaf blight"
print("Mapping updated. Now run the unpack cell and everything below it again.")
